# CNOT from Two iSWAPs - Physical DQD Implementation

**Physical setup (`DQDsystem` - `setup.py`):**
- `DQDsystem(tc, bx, Bz, Vac0, wc, gc, photon_max, epsilon_idle)` - energies in $\mu$ eV, frequencies in MHz
- Hilbert space: $\mathcal{H}_{\rm photon} \otimes \mathcal{H}_{\rm DQD1} \otimes \mathcal{H}_{\rm DQD2}  = n_{\rm photon,max} \times 4 \times 4$ dimensions
- Time unit: **ns**; frequencies in **GHz = rad/ns**; $\hbar = 0.6582\ \mu\text{eV}\cdot\text{ns}$

**Gate mechanisms (only `epsilon` is dynamically tunable; Bz, bx, tc are fixed):**
- **iSWAP**: `iswap_H(dqd)` - constant H at epsilon=0 sweet spot (cavity-mediated dispersive exchange)
- **Rx($\theta$)**: `xrot_H(dqd, target, t_start, t_end)`- EDSR drive $V_\mathrm{ac0}\cdot\tau_z\cdot\cos(E_\sigma/\hbar\cdot t)$ -> $\sigma_x$ via $\sin\bar\varphi$
- **Rz($\theta$)** `DQDSequenceCompiler.add_zrot()` - accumulates phase, offsets next EDSR drive
- Detuning cause phase difference between two qubits. Correction is needed

In [2]:
import numpy as np
from numpy import pi
import qutip as qt
from qutip import tensor, sigmax, sigmay, sigmaz, qeye, destroy
from qutip.qip.operations import cnot, iswap
from qutip import sprepost, to_chi, process_fidelity
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, ".")

from dqd import (
    DQDsystem,
    hbar_ns,
    iswap_H, xrot_H, zrot_H,
    zrot_time_ns, zrot_delta_freq_GHz,
    vacuum_eigenstates, initial_full_state,
    print_summary, plot_pulse_schedule, DQDSequenceCompiler,
    run_unitary, vacuum_subspace_unitary, two_qubit_unitary,
)
from dqd.circuit import _H0_ns, _eps1_op, _eps2_op, _edsr1_op, _edsr2_op

## Initialization

In [3]:
Pi2 = 2 * pi

dqd = DQDsystem(
    tc=40, bx=2, Bz=20, Vac0=1.0,
    wc=Pi2 * 4.862e3, gc=Pi2 * 50, photon_max=5, epsilon_idle=2000.0,
)
print_summary(dqd)

J_ns  = (dqd.g_sigma**2 / dqd.d_sigma) * 1e-3   # GHz (effective exchange)
r_s, r_t = dqd.dispersive_ratios()
print(f"\ng_sigma  = {dqd.g_sigma:.3f} MHz")
print(f"d_sigma  = {dqd.d_sigma:.3f} MHz")
print(f"J        = {J_ns*1e3:.4f} MHz")
print(f"g/Delta  = {r_s:.5f}")

DQD:        tc=40 ÃƒÅ½Ã‚Â¼eV,  Bz=20 ÃƒÅ½Ã‚Â¼eV (fixed),  bx=2 ÃƒÅ½Ã‚Â¼eV (fixed)
Cavity:     wc=4862 MHz,  gc=50.0 MHz
Qubit:      EÃƒÂÃ†â€™=19.9933 ÃƒÅ½Ã‚Â¼eV  (30.375 GHz)
ÃƒÂÃ¢â‚¬Â ÃƒÅ’Ã¢â‚¬Å¾:          1.527Ãƒâ€šÃ‚Â°,  sin(ÃƒÂÃ¢â‚¬Â ÃƒÅ’Ã¢â‚¬Å¾)=0.026656
Coupling:   g_ÃƒÂÃ†â€™=8.3742 MHz,  ÃƒÅ½Ã¢â‚¬Â_ÃƒÂÃ†â€™=-173.62 MHz,  g/ÃƒÅ½Ã¢â‚¬Â=0.04823
tg(iSWAP):  3888.9472 ns
t(Rx90):    38.7874 ns,  t(Rx180): 77.5748 ns
Vac0:       1.0 ÃƒÅ½Ã‚Â¼eV  (Rabi=40.498 MHz)
ÃƒÅ½Ã‚Âµ_idle:     2000.0 ÃƒÅ½Ã‚Â¼eV

g_sigma  = 8.374 MHz
d_sigma  = -173.621 MHz
J        = -0.4039 MHz
g/Delta  = 0.04823


### Gate Duration Calculation

In [4]:
# Derived gate parameters
Esigma_GHz  = dqd.Esigma / hbar_ns              # physical EDSR carrier [GHz]
phi_bar     = dqd.phi_bar
Omega_R     = dqd.Vac0 * np.sin(phi_bar) / hbar_ns   # Rabi angular freq [GHz]
t_pulse_x90 = pi / 2 * hbar_ns / (dqd.Vac0 * np.sin(phi_bar))   # Rx(pi/2) [ns]
tg_iSWAP    = dqd.iSWAP_gate_time() * 1e3       # iSWAP duration [ns]

EPS_Z       = 100.0                              # ueV - eps for physical Rz gates
delta_f_z   = zrot_delta_freq_GHz(dqd, EPS_Z)   # GHz

print(f"Esigma (carrier): {Esigma_GHz:.4f} GHz  -> period {2*pi/Esigma_GHz:.4f} ns")
print(f"phi_bar = {np.degrees(phi_bar):.3f} deg,  sin(phi_bar) = {np.sin(phi_bar):.6f}")
print(f"Omega_R (Rabi):  {Omega_R*1e3:.4f} mrad/ns  =  {Omega_R/2/pi*1e3:.4f} MHz")
print(f"t(Rx90):      {t_pulse_x90:.4f} ns")
print(f"t(iSWAP):     {tg_iSWAP:.2f} ns")
print(f"eps_Z = {EPS_Z} ueV")
print(f"Delta_fz = {delta_f_z*1e3:.4f} MHz")

# Single-DQD eigenenergies for rotating-frame correction
H_single = dqd.tc*dqd.tx + dqd.Bz/2*dqd.sz + dqd.bx/2*dqd.sx*dqd.tz
evals, _ = H_single.eigenstates(sort='low')
E_g_ns = float(evals[0]) / hbar_ns   # rad/ns
E_e_ns = float(evals[1]) / hbar_ns
print(f"\nSingle-DQD: E_g = {E_g_ns:.6f} GHz,  E_e = {E_e_ns:.6f} GHz")

Esigma (carrier): 30.3752 GHz  -> period 0.2069 ns
phi_bar = 1.527 deg,  sin(phi_bar) = 0.026656
Omega_R (Rabi):  40.4976 mrad/ns  =  6.4454 MHz
t(Rx90):      38.7874 ns
t(iSWAP):     3888.95 ns
eps_Z = 100.0 ueV
Delta_fz = 101.1336 MHz

Single-DQD: E_g = -75.978564 GHz,  E_e = -45.603338 GHz


In [5]:
seq = DQDSequenceCompiler(dqd)
seq.add_zrot(target=1, angle=-pi/2)   # virtual Rz(-pi/2) on Q1  [no pulse]
seq.add_xrot(target=2, angle=+pi/2)   # EDSR Rx(pi/2) on Q2      [physical]
seq.add_zrot(target=2, angle=+pi/2)   # virtual Rz(+pi/2) on Q2  [no pulse]
seq.add_iswap()                        # iSWAP #1                  [physical]
seq.add_xrot(target=1, angle=+pi/2)   # EDSR Rx(pi/2) on Q1      [physical]
seq.add_iswap()                        # iSWAP #2                  [physical]
seq.add_zrot(target=2, angle=+pi/2)   # virtual Rz(+pi/2) on Q2  [no pulse]

print(seq)
print(f"\nTotal sequence time: {seq.total_time:.4f} ns  =  {seq.total_time/1e3:.4f} us")

DQDSequenceCompiler  (4 steps, 7855.4692 ns total)
  Phase accumulators:  DQD1 = 2164.125Ãƒâ€šÃ‚Â°   DQD2 = 2434.125Ãƒâ€šÃ‚Â°
  [0.000 ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“ 38.787 ns]  Rx(90.0Ãƒâ€šÃ‚Â°)  target=2  ÃƒÂÃ¢â‚¬Â _drive=0.00Ãƒâ€šÃ‚Â°  Vac0=1.0 ÃƒÅ½Ã‚Â¼eV
  [38.787 ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“ 3927.735 ns]  iSWAP  (ÃƒÅ½Ã‚Âµ=0 both DQDs)
  [3927.735 ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“ 3966.522 ns]  Rx(90.0Ãƒâ€šÃ‚Â°)  target=1  ÃƒÂÃ¢â‚¬Â _drive=2164.12Ãƒâ€šÃ‚Â°  Vac0=1.0 ÃƒÅ½Ã‚Â¼eV
  [3966.522 ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“ 7855.469 ns]  iSWAP  (ÃƒÅ½Ã‚Âµ=0 both DQDs)

Total sequence time: 7855.4692 ns  =  7.8555 us


In [ ]:
from dqd.circuit import _eps1_op, _eps2_op
import scipy.linalg

H_td_A, _ = seq.compile()
H_mat      = H_td_A[0].full()

# Unified tlist over the full sequence:
#   xrot  -> fine grid (8 pts / EDSR cycle, ~0.026 ns)
#   iswap -> same dt so ODE takes ~1 internal sub-step per tlist point
#            (coarser dt is NOT faster: total sub-step count stays ~500k / iSWAP)
period_edsr = 2 * pi / Esigma_GHz
n_rx_pts    = max(2000, int(np.ceil(t_pulse_x90 / period_edsr)) * 8)
dt_rx       = t_pulse_x90 / n_rx_pts

offsets = [0.0]
for s in seq._steps:
    offsets.append(offsets[-1] + s['duration'])

parts = []
for i, step in enumerate(seq._steps):
    t0, t1 = offsets[i], offsets[i + 1]
    n = n_rx_pts if step['type'] == 'xrot' else max(2, round((t1 - t0) / dt_rx) + 1)
    parts.append(np.linspace(t0, t1, n))

tlist_full = np.unique(np.concatenate(parts))
n_isw_pts  = round(tg_iSWAP / dt_rx) + 1
print(f"tlist: {len(tlist_full):,} pts  "
      f"({n_rx_pts} per Rx,  {n_isw_pts:,} per iSWAP,  dt={dt_rx:.4f} ns)")
print("NOTE: iSWAP windows each require ~500k ODE sub-steps -> hours of runtime")

opts     = {"nsteps": 5_000_000, "atol": 1e-7, "rtol": 1e-7, "progress_bar": "tqdm"}
U_full_A = run_unitary(H_td_A, tlist_full, options=opts)
print("Done.")

# iSWAP matrix still needed for frame correction in c07
T_iswap     = tg_iSWAP
U_iswap_mat = scipy.linalg.expm(-1j * H_mat * T_iswap)

/home/topraramx/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tlist: 334,839 pts  (2000 per Rx,  165,421 per iSWAP,  dt=0.0138 ns)
NOTE: iSWAP windows each require ~500k ODE sub-steps -> hours of runtime


 66%|ÃƒÂ¢Ã¢â‚¬â€œÃ‹â€ ÃƒÂ¢Ã¢â‚¬â€œÃ‹â€ ÃƒÂ¢Ã¢â‚¬â€œÃ‹â€ ÃƒÂ¢Ã¢â‚¬â€œÃ‹â€ ÃƒÂ¢Ã¢â‚¬â€œÃ‹â€ ÃƒÂ¢Ã¢â‚¬â€œÃ‹â€ ÃƒÂ¢Ã¢â‚¬â€œÃ¢â‚¬Â¹   | 222474/334838 [27:49<10:56, 171.29it/s] 

In [ ]:
# --- Verify decomposition analytically (ideal gates, no coupling error) ---
def Rx_1q(theta): return (-.5j * theta * qt.sigmax()).expm()
def Rz_1q(theta): return (-.5j * theta * qt.sigmaz()).expm()

I = qt.qeye(2)

# Circuit applied right-to-left: first gate is rightmost
U_analytic = (
    qt.tensor(I,             Rz_1q(+pi/2)) *   # Q2: Rz(+pi/2)  [step 7 / virtual in Part A]
    iswap()                                 *   # iSWAP #2
    qt.tensor(Rx_1q(pi/2),  I            ) *   # Q1: Rx(pi/2)
    iswap()                                 *   # iSWAP #1
    qt.tensor(I,             Rz_1q(+pi/2)) *   # Q2: Rz(+pi/2)
    qt.tensor(I,             Rx_1q(pi/2) ) *   # Q2: Rx(pi/2)
    qt.tensor(Rz_1q(-pi/2), I            )     # Q1: Rz(-pi/2)
)

chi_ideal  = to_chi(sprepost(cnot(), cnot().dag()))
F_analytic = process_fidelity(
    to_chi(sprepost(U_analytic, U_analytic.dag())),
    chi_ideal)
print(f"Analytical decomposition fidelity vs CNOT: {F_analytic:.6f}")
print("(should be 1.0)")
print()
print("Analytical |U|:")
print(np.abs(U_analytic.full()).round(4))

In [ ]:
# --- Part A: Extract 4x4 qubit unitary and compute fidelity ---
#
# Frame correction strategy:
#   In the lab frame, U_full_A contains both the gate rotation AND the
#   background free evolution.  The background has eps_idle on the idle qubit
#   during each Rx window, shifting its eigenfrequency by ~1490 GHz relative
#   to eps=0.  Over 102 ns that is ~150 000 uncorrected radians - using the
#   simple diag(exp(+i E_ij * T)) formula gives completely wrong phases.
#
#   Correct approach: compute the background propagator U_free
#   (identical sequence, but with EDSR drives set to zero) via expm for
#   each constant-H segment, then extract the gate in the interaction picture:
#       U_gate = U_free_dag * U_full_A
#
# Segment Hamiltonians (no EDSR):
#   Rx_Q2 window : H_static + eps_idle * H_e1   (Q1 idled)
#   iSWAP windows: H_static
#   Rx_Q1 window : H_static + eps_idle * H_e2   (Q2 idled)

H_e1_mat = _eps1_op(dqd).full()
H_e2_mat = _eps2_op(dqd).full()
eps_idle  = dqd.epsilon_idle

H_bg_rx2 = H_mat + eps_idle * H_e1_mat   # during Q2 Rx, Q1 at eps_idle
H_bg_rx1 = H_mat + eps_idle * H_e2_mat   # during Q1 Rx, Q2 at eps_idle

U_bg_rx2 = scipy.linalg.expm(-1j * H_bg_rx2 * t_pulse_x90)
U_bg_rx1 = scipy.linalg.expm(-1j * H_bg_rx1 * t_pulse_x90)

# Compose free propagator in time order (rightmost = first applied):
# rx2 -> iswap1 -> rx1 -> iswap2
U_free_mat = U_iswap_mat @ U_bg_rx1 @ U_iswap_mat @ U_bg_rx2
U_free_Q   = qt.Qobj(U_free_mat, dims=H_td_A[0].dims)

# Interaction-picture gate
U_gate_full = U_free_Q.dag() * U_full_A
U_4x4_A     = two_qubit_unitary(U_gate_full, dqd)
U_corr_A    = qt.Qobj(U_4x4_A, dims=[[2, 2], [2, 2]])

F_A = process_fidelity(to_chi(sprepost(U_corr_A, U_corr_A.dag())), chi_ideal)
print(f"Part A (DQDSequenceCompiler, virtual Z):  F = {F_A:.6f}")
print("|U_corr_A|:")
print(np.abs(U_corr_A.full()).round(4))

In [ ]:
# --- Part A: DQDSequenceCompiler pulse schedule (DC eps, EDSR envelope, g_sigma) ---
from dqd import plot_pulse_schedule

fig = plot_pulse_schedule(
    dqd, seq,
    n_pts=3000,
    title="Part A CNOT - DQDSequenceCompiler (virtual Z, EDSR + iSWAP only)",
)
plt.show()

## Single-Gate Verification (circuit.py)

`circuit.py` provides per-gate Hamiltonian builders for standalone verification.
All functions take `dqd: DQDsystem` as first argument.

| Function | Returns | Purpose |
|----------|---------|---------|
| `iswap_H(dqd)` | `[H_const]` | iSWAP Hamiltonian at eps=0 |
| `xrot_H(dqd, target, t_start, t_end)` | `[H0, [EDSR, f(t)], [H_idle, g(t)]]` | EDSR Rx gate |
| `zrot_H(dqd, target, t_start, t_end, eps_amp)` | `[H0, [H_eps, f(t)], [H_idle, g(t)]]` | DC eps Rz gate |
| `initial_full_state(dqd, i, j)` | ket | $\|0\rangle_\text{photon} \otimes \|\text{eig}_i\rangle_\text{DQD1} \otimes \|\text{eig}_j\rangle_\text{DQD2}$ |

In [ ]:
from dqd import run_state, dqd_populations

# iSWAP gate (constant H at eps=0 sweet spot)
H_iswap_full = iswap_H(dqd)
psi0         = initial_full_state(dqd, dqd1_eigenstate_idx=0, dqd2_eigenstate_idx=1)
tlist_iswap  = np.linspace(0, tg_iSWAP, 2001)

print(f"iSWAP Hilbert space: {H_iswap_full[0].shape}")
print(f"iSWAP gate time:     {tg_iSWAP:.2f} ns")
print(f"Initial state dims:  {psi0.dims}")

# EDSR Rx(pi/2) on DQD1
H_x1_full = xrot_H(dqd, target=1, t_start=0.0, t_end=t_pulse_x90)
print(f"\nRx(90 deg) terms: {len(H_x1_full)}")
print(f"  [0] H0 static:    {H_x1_full[0].shape}")
print(f"  [1] EDSR tau_z1:  {H_x1_full[1][0].shape}")
print(f"  [2] eps_idle op:  {H_x1_full[2][0].shape}")
print(f"  Rx(90 deg) gate time: {t_pulse_x90:.4f} ns")

# DC eps Rz(pi/2) on DQD1
H_z1_full = zrot_H(dqd, target=1, t_start=0.0, t_end=t_sq_z, eps_amp=EPS_Z)
print(f"\nRz(90 deg): eps={EPS_Z} ueV -> Delta_f={zrot_delta_freq_GHz(dqd, EPS_Z)*1e3:.4f} MHz -> t={t_sq_z:.4f} ns")
print(f"  Rz(90 deg) terms: {len(H_z1_full)}")

print("\nTo run single-gate simulations:")
print("  U     = run_unitary(H_iswap_full, np.linspace(0, tg_iSWAP, 2001))")
print("  U_vac = vacuum_subspace_unitary(U)")
print("  U_4x4 = two_qubit_unitary(U, dqd)")